In [1]:
# NOTE: Set environment variables and import packages

import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"

import random
import time

import tqdm
import wandb
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
    SimpleReplayBuffer,
    save_params,
)

from fast_td3 import Critic, Actor

In [3]:
from fast_td3.hyperparams import HumanoidBenchArgs

args = HumanoidBenchArgs(
    env_name="h1-walk-v0",
    total_timesteps=20000,
    render_interval=5000,
    eval_interval=5000,
)
run_name = f"{args.env_name}_notebook_experiment"

In [4]:
# NOTE: GPU-Related Configurations

amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

Using device: cuda:0


In [ ]:
use_wandb = True

if use_wandb:
    wandb.init(
        project="FastTD3",
        name=run_name,
        config=vars(args),
        save_code=True,
    )

In [7]:
env_name = None
wandb_entity = "thuaduc24042001-technical-university-of-munich"
seed = 0

In [ ]:
import gymnasium as gym


def make_env(
    rank,
    seed=0
):
    """
    Utility function for multiprocessed env.

    :param rank: (int) index of the subprocess
    :param seed: (int) the inital seed for RNG
    """

    def _init():
        
        env = gym.make(env_name)
        env.action_space.seed(seed)
        
        return env

    return _init